In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from google.colab import drive
drive.mount('/content/drive')
import pickle

Mounted at /content/drive


In [ ]:
%run "/content/drive/MyDrive/Colab Notebooks/CombinedTrainingDataset.ipynb"
%run "/content/drive/MyDrive/Colab Notebooks/StopWords.ipynb"

100%|██████████| 2.09M/2.09M [00:00<00:00, 114MB/s]

Extracting files...


100%|██████████| 71.2M/71.2M [00:00<00:00, 172MB/s]

Extracting files...


Successfully combined!
Dataset 1 rows: 12854
Dataset 2 ('politics' only) rows: 25000
Total combined rows: 37854


In [ ]:
docs = combined_data['Combined_Content'].tolist()

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words=my_stop_words, min_df=20, token_pattern=r'(?u)\b[a-zA-Z]{3,}\b')

In [ ]:
tfidf_matrix = tfidf_vectorizer.fit_transform(docs)
num_topics = 50
nmf_model = NMF( n_components=num_topics, random_state=42, init='nndsvda', max_iter=500 )
nmf_W = nmf_model.fit_transform(tfidf_matrix)
nmf_H = nmf_model.components_

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['let'] not in stop_words.
  warnings.warn(


In [ ]:
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
n_top_words = 20

In [ ]:
tfidf_formatted_labels = []
for topic_idx, topic in enumerate(nmf_H):
    top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
    top_features = [tfidf_feature_names[i] for i in top_features_ind]
    clean_label = ", ".join(top_features)
    tfidf_formatted_labels.append(clean_label)

In [ ]:
print("\n--- TF-IDF Baseline Topics ---")
for i, label in enumerate(tfidf_formatted_labels):
    print(f"Topic {i}: {label}")

baseline_topic_list = tfidf_formatted_labels


--- TF-IDF Baseline Topics ---
Topic 0: automatically, performed, courteous, insults, wishing, shill, contact, advocating, permanent, concerns, harm, violation, users, hate, physical, ban, speech, attack, death, index
Topic 1: mueller, testify, conclusion, robert, bob, investigation, testimony, memo, phone, supported, knew, findings, wrote, words, team, straight, objected, bring, accurate, speak
Topic 2: https, com, www, org, status, html, wiki, wikipedia, youtu, removed, jpg, index, imgur, nytimes, message, view, redd, amp, google, following
Topic 3: barr, lied, william, credibility, testimony, perjury, impeached, tomorrow, cover, memo, summaries, resign, lies, hearing, inaccurate, decide, officials, oath, impeach, trouble
Topic 4: lot, come, long, change, exactly, live, help, hard, little, life, different, god, ago, start, kind, dude, mind, gonna, climate, bullshit
Topic 5: graham, lindsey, lindsay, emails, hearing, feinstein, senator, opening, msnbc, piece, bullshit, males, swear, 

In [ ]:
topic_id_to_label = {
    0: "Noise/Filler",
    1: "Mueller Investigation Findings",
    2: "URLs & Web Metadata",
    3: "Bill Barr Perjury Allegations",
    4: "Life & Big Questions",
    5: "Senate Hearing Debates",
    6: "Mueller Report Release & Redactions",
    7: "Russia-Ukraine Invasion & NATO",
    8: "Special Counsel Public Context",
    9: "House Oversight & Subpoenas",
    10: "Fighting for Accountability",
    11: "Fox, CNN & Media Bias",
    12: "Congress vs. The Law",
    13: "Political Talking Points & Emails",
    14: "How Politics Works",
    15: "Capitalism vs. Socialism Debate",
    16: "Reddit Community Feedback",
    17: "NY Politics & Green New Deal",
    18: "Senate Majority & McConnell",
    19: "Identity Politics & Extremism",
    20: "Leaked Letters & Documents",
    21: "Arguing over Definitions",
    22: "Socialism & Welfare",
    23: "War & Anti-War Views",
    24: "DOJ Charges & Statements",
    25: "Impeachment Trial",
    26: "Health Care & Insurance",
    27: "Campaign Finance & Economic Costs",
    28: "Primary Elections & Voter Base",
    29: "Jobs & Fair Pay",
    30: "Power & Influences",
    31: "Russia Sanctions & Collusion",
    32: "Political Skepticism",
    33: "Strong Public Reactions",
    34: "Critique of News Coverage",
    35: "Supreme Court & Judges",
    36: "Barr's Report Summary",
    37: "Race & Civil Rights History",
    38: "Approval Ratings & Polls",
    39: "Progressive & Socialist Sentiment",
    40: "Rating the Attorney General",
    41: "Taxes & The Wealthy",
    42: "Violence & Extremism",
    43: "Law Enforcement & Abortion Rights",
    44: "Calls for Resignation & Action",
    45: "Legal Rights & Lying",
    46: "Unions & Strikes",
    47: "Obstruction & Collusion",
    48: "Lying Under Oath",
    49: "China & Communism"
}

In [ ]:
dominant_topics = nmf_W.argmax(axis=1)
results_df = pd.DataFrame({
    'Document_Content': docs,
    'Topic_ID': dominant_topics
})

In [ ]:
results_df['Topic_Label'] = results_df['Topic_ID'].map(topic_id_to_label)
print(results_df[['Document_Content', 'Topic_Label']])

                                        Document_Content  \
0      No matter who someone is, how they look like, ...   
1       Biden speech draws 38.2 million U.S. TV viewers    
2      State of the union Who watched the state of th...   
3                 We Should Just Give Poor People Money    
4                                     Do it for the Dew    
...                                                  ...   
37849  Everyone who cares about truth and justice is ...   
37850  That is a big question.  The Senate has never ...   
37851  The report literally says the President was no...   
37852  Ok now let´s play a unfun little game:\n\nBarr...   
37853                       Yay this solves the problem!   

                             Topic_Label  
0                       Unions & Strikes  
1                           Noise/Filler  
2                       Unions & Strikes  
3      Campaign Finance & Economic Costs  
4                           Noise/Filler  
...                  

In [ ]:
save_path = '/content/drive/MyDrive/tfidf_topic_model.pkl'
with open(save_path, 'wb') as f:
    pickle.dump({
        'vectorizer': tfidf_vectorizer,
        'nmf_model': nmf_model,
        'top_words_labels': tfidf_formatted_labels,
        'human_labels': topic_id_to_label
    }, f)
print("topic model saved as 'tfidf_topic_model.pkl'")

topic model saved as 'tfidf_topic_model.pkl'
